# Environmental justice: pipeline incidents vs social vulnerability

Do natural-gas and liquid pipeline incidents cluster in more vulnerable
communities? This notebook builds a state-level comparison between
PHMSA pipeline incident counts and the CDC Social Vulnerability Index
(SVI), then scatters them.

Both datasets are public and tracked in Sondio as
`sondio.us.phmsa.pipeline_incidents` and `sondio.us.cdc.svi_tracts`.

In [ ]:
%pip install --quiet 'sondio[geo]>=0.1,<0.2' matplotlib

import os
import sondio

# Colab: stores SONDIO_API_KEY in the Secrets panel (🔑 sidebar).
# Local: reads the env var or ~/.sondio/config.
try:
    from google.colab import userdata
    sondio.api_key = userdata.get("SONDIO_API_KEY")
except (ImportError, Exception):
    sondio.api_key = os.environ.get("SONDIO_API_KEY", "sk_sondio_your_key_here")
print(f"sondio {sondio.__version__}")

In [ ]:
# All PHMSA pipeline incidents we have in the dataset (multi-year).
incidents = sondio.us.phmsa.pipeline_incidents(all_pages=True)
print(f"{len(incidents):,} incidents")
incidents[["operator_name", "state", "incident_date", "total_cost_current"]].head()

In [ ]:
# State-level tract summary: mean RPL (overall social-vulnerability rank,
# 0 = least vulnerable, 1 = most) and total population. SVI has ~84k
# tracts and we only need the state rollup, so pull the most recent year.
import pandas as pd

tracts = sondio.us.cdc.svi_tracts(data_year=2022, all_pages=True)
print(f"{len(tracts):,} SVI tracts")
state_svi = (
    tracts.dropna(subset=["rpl_themes_overall", "total_population"])
    .groupby("subdivision_code")
    .apply(lambda g: pd.Series({
        "mean_rpl": (g["rpl_themes_overall"] * g["total_population"]).sum() / g["total_population"].sum(),
        "population": g["total_population"].sum(),
    }))
)
state_svi.head()

In [ ]:
# Incidents per million residents — normalize so small states aren't
# drowned out by TX and CA.
state_incidents = incidents.groupby("state").size().rename("incidents")
state = state_svi.join(state_incidents).dropna()
state["per_million"] = state["incidents"] / (state["population"] / 1_000_000)
state.sort_values("per_million", ascending=False).head(10)

In [ ]:
# Scatter: mean population-weighted SVI vs incidents-per-million.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(state["mean_rpl"], state["per_million"], s=60, alpha=0.7, color="#c44")
for code, row in state.iterrows():
    ax.annotate(code, (row["mean_rpl"], row["per_million"]), fontsize=8, xytext=(3, 2), textcoords="offset points")
ax.set_xlabel("Mean social-vulnerability rank (population-weighted)")
ax.set_ylabel("Pipeline incidents per 1M residents")
ax.set_title("State-level SVI vs pipeline incidents (PHMSA)")
ax.grid(alpha=0.2); plt.tight_layout(); plt.show()

## Reading the chart

Each dot is a state. A state with **higher mean RPL** has, on average,
more socially vulnerable census tracts. A state high on the y axis has
more pipeline incidents per capita.

This rollup is deliberately coarse. A real environmental-justice
analysis would join incidents to their containing tract directly — once
the incidents table carries lat/lon consistently. The state-level view
is the starting point.

## Next steps
- Narrow to a single vulnerable state (e.g. `state="LA"`) and chart
  incident density at the tract level.
- Replace the SVI rollup with the `rpl_theme1_socioeconomic` theme if
  you care specifically about income + education, not the composite.
- Cross with `sondio.oilgas.wells(...)` to show well density in the
  same tracts.